In [1]:
%load_ext autoreload
%autoreload 2


# kala77 andmebaasi tegemine

Skripti töö tulemusena tekitatakse uus andmebaasifail **kala77**.
Baas sisaldab kõiki transaktsioone, mis pole kaetud olemasolevate mustritega.
Kaetud on **ainult mustribaasis olevad** verbid.

Sisendiks on:

- verbi mustrite andmebaas: verb_patterns_new.db
- verbi transaktsioonide andmebaas: v33_transactions.db

## andmebaas kala77

### Tabelid

### TABEL1 transaction_head

| väli          | tüüp | kirjeldus                                                                         | näide      | märkus          |
| ------------- | ---- | --------------------------------------------------------------------------------- | ---------- | --------------- |
| id            | int  | rea <br/>unikaalne ID                                                             | _56_       |                 |
| sentence_id   | int  | lause id andmebaasis                                                              |            |                 |
| loc           | int  | verbi asukoht lauses                                                              |            |                 |
| verb          | text | verbi lemma                                                                       | _olema_    |                 |
| verb_compound | text | verbi afiksaaladverbid                                                            | alla,peale | eraldajaks koma |
| form          | text | verb sellises vormis, nagu see lauses esines                                      | _oli_      |                 |
| deprel        | text | verbi deprel                                                                      |            |                 |
| feats         | text | verbi morf kategooriad alfabeetilises järjekorras                                 | aux,ps3    |                 |
| phrase        | text | puhastatud fraas (ainult need alluvad, mis on transactions tabelisse salvestatud) |            |                 |

### TABEL2 transaction

| väli       | tüüp | kirjeldus                                                            | näide  | märkus |
| ---------- | ---- | -------------------------------------------------------------------- | ------ | ------ |
| id         | int  | rea <br/>unikaalne ID                                                | _56_   |        |
| head_id    | int  | rea transaction_head.id                                              |        |        |
| loc        | int  | sõna asukoht lauses                                                  |        |        |
| loc_rel    | int  | sõna asukoht verbi suhtes                                            |        |        |
| deprel     | text | sõna deprel                                                          |        |        |
| form       | text | sõna vorm                                                            |        |        |
| lemma      | text | sõna lemma                                                           |        |        |
| pos        | text | sõna sõnaliik                                                        |        |        |
| feats      | text | sõna morf kategooriad alfabeetilises järjekorras                     | add,sg |        |
| parent_loc | int  | vanema tipu loc, juhul kui tegemist on <code>obl</obl> alluvaga case | 2      |        |

### TABEL3 verbs_table

| väli          | tüüp | kirjeldus                               | näide    | märkus          |
| ------------- | ---- | --------------------------------------- | -------- | --------------- |
| verb_id       | int  | verbi unikaalne id                      |          |                 |
| verb          | text | verbi lemma                             | _aasima_ |                 |
| verb_compound | text | verbi afiksaaladverbid                  |          | eraldajaks koma |
| pat_ids       | text | mustrite id-d verb_patterns andmebaasis |          | eraldajaks koma |

### TABEL4 verb_transactions

| väli    | tüüp | kirjeldus         | näide | märkus |
| ------- | ---- | ----------------- | ----- | ------ |
| verb_id | int  | verbi id          |       |        |
| head_id | int  | transaktsiooni id |       |        |

### TABEL5 patterns

| väli          | tüüp | kirjeldus | näide | märkus |
| ------------- | ---- | --------- | ----- | ------ |
| pat_id        | int  |           |       |        |
| pattern       | text |           |       |        |
| verb_word     | text |           |       |        |
| verb_compound | text |           |       |        |
| phrase_nr     | int  |           |       |        |
| phrase_case   | text |           |       |        |
| adp           | text |           |       |        |
| inf_verb      | text |           |       |        |


In [3]:
import sys
sys.path.append('../../../common_code')
from paths import PATH_ROOT

from db_operations.db_table_ops import copy_table_structure
from db_operations.verb_transactions.filter_verb_transaction_tables import filter_verb_transaction_tables

from kala77_helpers import *


PATH_TRANSACTIONS_DB = PATH_ROOT + "/databases/v33_data.db"
PATH_PATTERNS_DB = PATH_ROOT + "/databases/vp_data3.db"
KALA77_DB = PATH_ROOT + "/databases/kala77.db"

In [4]:
# tekitab andmebaasifaili, kui seda veel ei olnud
import sqlite3
from sqlalchemy import create_engine

con = sqlite3.connect(KALA77_DB)
con.close()

DATABASE_PATH = f"sqlite:///{KALA77_DB}"

engine = create_engine(DATABASE_PATH)
reset_tables(engine)
conn = engine.connect()
conn.close()


# kasutame sqlalchemyt, et oleks lihtsam tabeleid luua
conn = sqlite3.connect(KALA77_DB)

conn.row_factory = sqlite3.Row

# liidame teised andmebaasid
conn.execute(f"ATTACH DATABASE '{PATH_PATTERNS_DB}' AS db_pat")
conn.execute(f"ATTACH DATABASE '{PATH_TRANSACTIONS_DB}' AS db_tr")

In [5]:
%%time
# täidame verbs tabeli
fill_table_verbs(conn=conn)

# kopeerime mustride tabeli andmetega
copy_table_structure(conn, table_name='db_pat.patterns', new_table_name='patterns', copy_data=True, delete_if_exists=True, verbose=False)

CPU times: user 3.12 ms, sys: 2.45 ms, total: 5.57 ms
Wall time: 7.07 ms


('main', 'patterns')

In [6]:
%%time
verbs = conn.execute(
    "SELECT verb_id, pat_ids, verb, verb_compound FROM verbs;"
    ).fetchall()
verbs = [dict(row) for row in verbs]

print(f"andmebaasi lisati {len(verbs)} verbi")

andmebaasi lisati 1271 verbi
CPU times: user 3.26 ms, sys: 1.19 ms, total: 4.45 ms
Wall time: 3.75 ms


In [ ]:
%%time
# täidame verb_transactions tabeli
from tqdm import tqdm
for v in tqdm(verbs):
    fill_table_verb_transactions(conn=conn, verb_id=v['verb_id'], pat_ids=v['pat_ids'].split(','))
conn.execute("SELECT COUNT(verb_id) FROM verb_transactions").fetchone()[0]

100%|██████████| 3/3 [00:00<00:00, 29.39it/s]

CPU times: user 12.4 ms, sys: 18.2 ms, total: 30.5 ms
Wall time: 121 ms


4085

In [9]:
%%time 
# tekitab uued transaction_head ja transaction_row tabelid, võtab ca paarkümmend minutit aega
# kopeeritakse tabelite struktuur db_tr andmebaasist
# täidetakse sisuga, tehakse join verb_transactions.head_id tabeliga
filter_verb_transaction_tables(
    conn=conn,
    source_schema='db_tr',
    transaction_head='transaction_head',
    transaction_row='transaction_row',
    ids_table='verb_transactions',
    ids_column='head_id',
    delete_if_exists=True,
    copy_indexes=True,
    verbose=False,
)


CPU times: user 1.6 s, sys: 825 ms, total: 2.42 s
Wall time: 4.03 s


True

In [10]:
#%%time
# kontrollime 10 juhusliku verbi pealt, et numbrid jooksevad kokku
#import random
#random_i = random.sample(range(0, len(verbs)-1), 10)
# check verbs stat
# get counts of random transactions to check, that numbers align together
#for v in [verbs[i] for i in random_i]:
#    show_verb_trans_stat(conn=conn, verb=v)
    

In [9]:
conn.close()